In [1]:
import os
import mlflow
import requests
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier


In [2]:
## download the dataset
# Directory of the raw data files
_data_root = './data/covertype'
# Path to the raw training data
_data_filepath = os.path.join(_data_root, 'covertype_train.csv')
# Download data
os.makedirs(_data_root, exist_ok=True)
if not os.path.isfile(_data_filepath):
    #https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/
    url = 'https://docs.google.com/uc?export= \
    download&confirm={{VALUE}}&id=1lVF1BCWLH4eXXV_YOJzjR7xZjj-wAGj9'
    r = requests.get(url, allow_redirects=True, stream=True)
    open(_data_filepath, 'wb').write(r.content)

In [3]:
# Load the dataset to a dataframe
df = pd.read_csv(_data_filepath)

# Set the target values
y = df['Cover_Type']#.values

# Set the input values
df.drop('Cover_Type', axis=1, inplace=True)
X = df#.values

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [4]:
column_trans = make_column_transformer((OneHotEncoder(handle_unknown='ignore'),
                                        ["Wilderness_Area", "Soil_Type"]),
                                      remainder='passthrough') # pass all the numeric values through the pipeline without any changes.

column_trans


ColumnTransformer(remainder='passthrough',
                  transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['Wilderness_Area', 'Soil_Type'])])

In [5]:
pipe = Pipeline(steps=[("column_trans", column_trans),("scaler", StandardScaler(with_mean=False)), ("RandomForestClassifier", RandomForestClassifier())])

pipe

Pipeline(steps=[('column_trans',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Wilderness_Area',
                                                   'Soil_Type'])])),
                ('scaler', StandardScaler(with_mean=False)),
                ('RandomForestClassifier', RandomForestClassifier())])

In [6]:
param_grid =  {'RandomForestClassifier__max_depth': [1,2,3,10], 'RandomForestClassifier__n_estimators': [10,11]}

search = GridSearchCV(pipe, param_grid, n_jobs=2)
search


GridSearchCV(estimator=Pipeline(steps=[('column_trans',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('onehotencoder',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['Wilderness_Area',
                                                                          'Soil_Type'])])),
                                       ('scaler',
                                        StandardScaler(with_mean=False)),
                                       ('RandomForestClassifier',
                                        RandomForestClassifier())]),
             n_jobs=2,
             param_grid={'RandomForestClassifier__max_depth': [1, 2, 3, 10],
                         'RandomForestClassifier__n_estimators': [10, 11]})

In [7]:
search.get_params().keys()

dict_keys(['cv', 'error_score', 'estimator__memory', 'estimator__steps', 'estimator__verbose', 'estimator__column_trans', 'estimator__scaler', 'estimator__RandomForestClassifier', 'estimator__column_trans__n_jobs', 'estimator__column_trans__remainder', 'estimator__column_trans__sparse_threshold', 'estimator__column_trans__transformer_weights', 'estimator__column_trans__transformers', 'estimator__column_trans__verbose', 'estimator__column_trans__verbose_feature_names_out', 'estimator__column_trans__onehotencoder', 'estimator__column_trans__onehotencoder__categories', 'estimator__column_trans__onehotencoder__drop', 'estimator__column_trans__onehotencoder__dtype', 'estimator__column_trans__onehotencoder__feature_name_combiner', 'estimator__column_trans__onehotencoder__handle_unknown', 'estimator__column_trans__onehotencoder__max_categories', 'estimator__column_trans__onehotencoder__min_frequency', 'estimator__column_trans__onehotencoder__sparse', 'estimator__column_trans__onehotencoder__s

In [8]:
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

import os
#os.environ['MLFLOW_S3_ENDPOINT_URL'] = "http://10.43.101.149:9000"
#os.environ['AWS_ACCESS_KEY_ID'] = 'admin'
#os.environ['AWS_SECRET_ACCESS_KEY'] = 'supersecret'

# connect to mlflow
#mlflow.set_tracking_uri("http://10.43.101.149:5000")
mlflow.set_experiment("mlflow_tracking_examples")

mlflow.autolog(log_model_signatures=True, log_input_examples=True, log_models=True)

with mlflow.start_run(run_name="autolog_with_pipeline") as run:
    search.fit(X_train, y_train)

2026/03/18 00:55:02 INFO mlflow.tracking.fluent: Experiment with name 'mlflow_tracking_examples' does not exist. Creating a new experiment.
2026/03/18 00:55:02 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.1.post1 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/03/18 00:55:02 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/18 00:55:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid

🏃 View run brawny-sheep-42 at: http://mlflow:5000/#/experiments/3/runs/01ddea8c969342b297e09086fbbe50ae
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run marvelous-ant-674 at: http://mlflow:5000/#/experiments/3/runs/15e0272ff3e346fdbb30e9f7771ef57c
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run colorful-shrew-532 at: http://mlflow:5000/#/experiments/3/runs/3e23c3954a0d41a892758bcbac266951
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run bouncy-moose-55 at: http://mlflow:5000/#/experiments/3/runs/07b270e41d6849d0b11ee6891f583153
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run selective-stoat-162 at: http://mlflow:5000/#/experiments/3/runs/1b7592fbdb274ab6bbd76cd36a8d1fec
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run autolog_with_pipeline at: http://mlflow:5000/#/experiments/3/runs/b9c8e6ace03343a98f73d4ee6fcf4272
🧪 View experiment at: http://mlflow:5000/#/experiments/3


In [11]:
print('tracking uri:', mlflow.get_tracking_uri())
print('artifact uri:', mlflow.get_artifact_uri())

tracking uri: http://mlflow:5000
artifact uri: s3://mlflows3/artifacts/3/a98cb37408e64afeb84ec9cb9e2a1e1a/artifacts


In [12]:
mlflow.end_run()

🏃 View run popular-shrew-146 at: http://mlflow:5000/#/experiments/3/runs/a98cb37408e64afeb84ec9cb9e2a1e1a
🧪 View experiment at: http://mlflow:5000/#/experiments/3


In [13]:
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

import os

mlflow.sklearn.autolog(log_model_signatures=True, log_input_examples=True, registered_model_name="modelo1", log_models=True)

with mlflow.start_run(run_name="autolog_pipe_model_reg") as run:
    search.fit(X_train, y_train)

2026/03/18 01:02:18 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.1.post1 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/03/18 01:02:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing va

🏃 View run adorable-hog-925 at: http://mlflow:5000/#/experiments/3/runs/64f03c8ce4584f8586d143f875897568
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run upbeat-eel-341 at: http://mlflow:5000/#/experiments/3/runs/7fdd57ff9d14456f83f8b0a5be8f921d
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run handsome-calf-519 at: http://mlflow:5000/#/experiments/3/runs/4ad8c132af4045ceadadecbc5a3ef724
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run luminous-hound-70 at: http://mlflow:5000/#/experiments/3/runs/e509dd0273e04f69b3ae4426740d98dd
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run rebellious-cat-680 at: http://mlflow:5000/#/experiments/3/runs/48fc0e9fc9704b1b9bbcfb8cb5aa649d
🧪 View experiment at: http://mlflow:5000/#/experiments/3
🏃 View run autolog_pipe_model_reg at: http://mlflow:5000/#/experiments/3/runs/6972c976b202444ba6e58b06c4df031e
🧪 View experiment at: http://mlflow:5000/#/experiments/3


In [14]:
import mlflow

import os

model_name = "modelo1"

# logged_model = 'runs:/71428bebed2b4feb9635714ea3cdb562/model'
model_production_uri = "models:/{model_name}/production".format(model_name=model_name)

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(model_uri=model_production_uri)
loaded_model
example_test = X_test.iloc[0].to_frame().T
#print(example_test)
print('real: ', y_test.iloc[0])
print('prediction: ', loaded_model.predict(example_test))

2026/03/18 01:03:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


real:  0
prediction:  [1]


In [15]:
X_test.iloc[0].to_frame().T

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type
34125,3266,208,23,295,95,3082,197,254,182,808,Commanche,C7757


In [16]:
print('real: ', y_test.iloc[0])
print('prediction: ', loaded_model.predict(example_test))

2026/03/18 01:03:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


real:  0
prediction:  [1]


In [ ]:
m-d4814fd0e0fe4c2fadfa4199e170da0d

In [19]:


# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(model_uri="s3://mlflows3/artifacts/1/models/m-d4814fd0e0fe4c2fadfa4199e170da0d/artifacts")
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: s3://mlflows3/artifacts/1/models/m-d4814fd0e0fe4c2fadfa4199e170da0d/artifacts
  flavor: mlflow.sklearn
  run_id: 6ca2afa5ce4749fc90fa2d813313add0